# Prithvi TC Inference

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [2]:
%ls -ltrh

total 12K
-rw------- 1 simon simon  790 Nov 16 20:34 model.toml
-rw-rw-r-- 1 simon simon 6.7K Nov 16 20:56 prithvi_tc_inference.ipynb


In [3]:
%env PRITHVI_DATA_PATH=/data1/prithvi_precip/data/scaling_factors

env: PRITHVI_DATA_PATH=/data1/prithvi_precip/data/scaling_factors


In [4]:
from pytorch_retrieve.architectures import load_and_compile_model
mdl = load_and_compile_model("model.toml")

In [5]:
from pytorch_retrieve.training import load_weights

load_weights(mdl, "/home/simon/src/fm4a/data/weights/

In [34]:
import torch
from typing import Dict

def post_process_results(inpt: Dict[str, torch.Tensor], results: Dict[str, torch.Tensor]) -> xr.Dataset:
    """
    Extracts surface winds, surface pressure and 850 winds from forecast results.

    Args:
        inpt: The batch containing the input data.
        results: A dictionary containing the forecast results.
    """
    static = inpt["static"]
    if static.dim() == 5:
        lats = np.rad2deg(inpt["static"][0, 0, 0, :, 0].float().cpu().numpy())
        lons = np.rad2deg(inpt["static"][0, 0, 1, 0, :].float().cpu().numpy())
    else:
        lats = np.rad2deg(inpt["static"][0, 0, :, 0].float().cpu().numpy())
        lons = np.rad2deg(inpt["static"][0, 1, 0, :].float().cpu().numpy())

    dataset = xr.Dataset({
        "latitude": (("latitude",), lats),
        "longitude": (("longitude",), lons)
    })

    pred = results["y"]
    pred = np.stack([step[:, [9, 17, 18, -18 -4]].float().cpu().numpy() for step in pred], axis=1)
    slp = pred[:, :, 0].float().cpu().numpy()
    u10 = pred[:, :, 1].float().cpu().numpy()
    v10 = pred[:, :, 2].float().cpu().numpy()
    u850 = pred[:, :, -18].float().cpu().numpy()
    v850 = pred[:, :, -4].float().cpu().numpy()

    dataset["slp"] = (("batch", "step", "latitude", "longitude"), slp)
    dataset["u10"] = (("batch", "step", "latitude", "longitude"), u10)
    dataset["v10"] = (("batch", "step", "latitude", "longitude"), v10)
    dataset["u850"] = (("batch", "step", "latitude", "longitude"), u850)
    dataset["v850"] = (("batch", "step", "latitude", "longitude"), v850)
    return dataset


In [35]:
from prithvi_precip.forecast.data_loaders import AutoregressiveForecastLoader

In [36]:
data_loader = AutoregressiveForecastLoader(
    "/data1/prithvi_precip/data/training_data/",
    init_times = np.array([np.datetime64("2021-08-27")]),
    n_steps=20,
    input_time=6,
    center_meridionally=False,
)
len(data_loader)

1

In [ ]:
from prithvi_precip.forecast.runners import run_autoregressive_forecast

run_autoregressive_forecast(
    mdl,
    data_loader,
    "/data1/prithvi_precip/results/prithvi_tc",
    post_process_fn=post_process_results
)

  0%|                                                                                                                | 0/1 [00:00<?, ?it/s]

## Visualize Results

In [ ]:
result_files = sorted(list(Path("/data1/prithvi_precip/results/prithvi_tc").glob("*.nc")))

In [ ]:
result_files